## Laboratorio 3: Detección de malware

Primero vamos a determinar si los archivos están empaquetados

In [33]:
import pefile
import os
import subprocess
import pandas as pd
import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
def show_sections(pefile_object):
    for section in pefile_object.sections:
        print("Nombre:", section.Name)
        print("VirtualAddress:", hex(section.VirtualAddress))
        print("Size:", section.SizeOfRawData)
        print("-" * 20)


carpeta = "MALWR"
archivos_procesados = 0

for archivo in os.listdir(carpeta):
    ruta = os.path.join(carpeta, archivo)

    # Solo archivos normales
    if os.path.isfile(ruta):
        try:
            print("\n" + "="*60)
            print(f"Analizando: {archivo}")
            print("="*60)

            pe = pefile.PE(ruta)
            show_sections(pe)

            archivos_procesados += 1

            if archivos_procesados >= 5:
                break

        except Exception as e:
            print(f"No es un PE válido: {archivo}")

if archivos_procesados == 0:
    print("No se encontraron ejecutables PE válidos.")



Analizando: F6655E39465C2FF5B016980D918EA028
Nombre: b'.text\x00\x00\x00'
VirtualAddress: 0x1000
Size: 3584
--------------------
Nombre: b'.rdata\x00\x00'
VirtualAddress: 0x2000
Size: 1536
--------------------
Nombre: b'.data\x00\x00\x00'
VirtualAddress: 0x3000
Size: 512
--------------------
Nombre: b'.rsrc\x00\x00\x00'
VirtualAddress: 0x4000
Size: 512
--------------------

Analizando: FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2
Nombre: b'.text\x00\x00\x00'
VirtualAddress: 0x1000
Size: 4096
--------------------
Nombre: b'.rdata\x00\x00'
VirtualAddress: 0x2000
Size: 2048
--------------------
Nombre: b'.data\x00\x00\x00'
VirtualAddress: 0x3000
Size: 512
--------------------
Nombre: b'.rsrc\x00\x00\x00'
VirtualAddress: 0x4000
Size: 512
--------------------

Analizando: 65018CD542145A3792BA09985734C12A
Nombre: b'.text\x00\x00\x00'
VirtualAddress: 0x1000
Size: 4096
--------------------
Nombre: b'.rdata\x00\x00'
VirtualAddress: 0x2000
Size: 2048
--------------------
Nombre: b'.data\x00\x00\x00

Vemos que si están empaquetados, vamos a desempaquetarlos

In [34]:
for file in os.listdir('MALWR'):
    route = os.path.join('MALWR', file)
    result = subprocess.run(['upx', '-d', route], capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f' {file} comprimido')
    else:
        print(f'{file} falló: {result.stderr}')

F6655E39465C2FF5B016980D918EA028 falló: upx: MALWR/F6655E39465C2FF5B016980D918EA028: NotPackedException: not packed by UPX

FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2 falló: upx: MALWR/FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2: NotPackedException: not packed by UPX

65018CD542145A3792BA09985734C12A falló: upx: MALWR/65018CD542145A3792BA09985734C12A: NotPackedException: not packed by UPX

6FAA4740F99408D4D2DDDD0B09BBDEFD falló: upx: MALWR/6FAA4740F99408D4D2DDDD0B09BBDEFD: NotPackedException: not packed by UPX

F8437E44748D2C3FCF84019766F4E6DC falló: upx: MALWR/F8437E44748D2C3FCF84019766F4E6DC: NotPackedException: not packed by UPX

B98hX8E8622C393D7E832D39E620EAD5D3B49 falló: upx: MALWR/B98hX8E8622C393D7E832D39E620EAD5D3B49: NotPackedException: not packed by UPX

SAM_B659D71AE168E774FAAF38DB30F4A84 falló: upx: MALWR/SAM_B659D71AE168E774FAAF38DB30F4A84: NotPackedException: not packed by UPX

POL55_A4F1ECC4D25B33395196B5D51A06790 falló: upx: MALWR/POL55_A4F1ECC4D25B33395196B5D51A06790: NotPa

Ya desempaquetados, vamos a crear el dataset, este tendrá estas características:

- Hash del archivop
- Imports
- Timestamp de compliación


In [35]:
def get_imports(path):
    """
    Extract function calls from header
    """
    imports = set()
    timestamp = None
    try:
        pe = pefile.PE(path)
        timestamp = pe.FILE_HEADER.TimeDateStamp

        if hasattr(pe, "DIRECTORY_ENTRY_IMPORT"):
            for entry in pe.DIRECTORY_ENTRY_IMPORT:
                dll = entry.dll.decode(errors="ignore")

                for imp in entry.imports:
                    if imp.name:
                        func = imp.name.decode(errors="ignore")
                        imports.add(f"{dll}:{func}")

    except pefile.PEFormatError:
        return set(), None

    except Exception:
        return set(), None
    
    return imports, timestamp


df =  pd.DataFrame(columns=['hash', 'imports', 'timestamp'])

for file in os.listdir('MALWR'):
    route = os.path.join('MALWR', file)
    imports, timestamp = get_imports(route)

    imports = ' '.join(list(imports))

    new_row = pd.DataFrame([{'hash': file, 'imports': imports, 'timestamp': timestamp }])

    df = pd.concat([df, new_row], ignore_index=True)
    
    if result.returncode == 0:
        print(f' {file} comprimido')
    else:
        print(f'{file} falló: {result.stderr}')


df.to_csv('raw_data.csv')

F6655E39465C2FF5B016980D918EA028 falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2 falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

65018CD542145A3792BA09985734C12A falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

6FAA4740F99408D4D2DDDD0B09BBDEFD falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

F8437E44748D2C3FCF84019766F4E6DC falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

B98hX8E8622C393D7E832D39E620EAD5D3B49 falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

SAM_B659D71AE168E774FAAF38DB30F4A84 falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14FC7E: NotPackedException: not packed by UPX

POL55_A4F1ECC4D25B33395196B5D51A06790 falló: upx: MALWR/QW2_4C6BDDCCA2695D6202DF38708E14

## Preprocesamiento

Vamos a usar TF-IDF para convertir la secuencia de imports a vectores

In [36]:
df_malware = pd.read_csv('raw_data.csv', index_col = 0)

print('Numero de filas: ', df_malware.shape[0])

# Fit and tran

Numero de filas:  41


In [37]:
df_malware.isna().sum()

hash         0
imports      1
timestamp    1
dtype: int64

Vemos que tenemos un archivo del cuál no hay ni imports ni timestamp, vamos a borrarlo pues la forma en la que pensamos analizar el malware es en base a los imports en la sección del header.

In [38]:
df_malware = df_malware.dropna(subset=['imports'])

df_malware.isna().sum()

hash         0
imports      0
timestamp    0
dtype: int64

In [40]:
def count_unique_imports(df, column="imports"):
    unique_imports = set()
    
    for row in df[column].dropna():
        imports = row.split()
        unique_imports.update(imports)
    
    return len(unique_imports), unique_imports

n_imports, imports_set = count_unique_imports(df_malware)

print("Número de imports únicos:", n_imports)

Número de imports únicos: 393


Y ahora vamos a vectorizar con TF-IDF

In [42]:
tfidf_vectorizer = TfidfVectorizer()

corpus = list(df_malware['imports'])

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

df_new = pd.DataFrame({
    'Hash': df_malware['hash'],
    'Imports': list(tfidf_matrix.toarray()),
    'Timestamp': df_malware['timestamp']
})

print(df_new.head())



                                       Hash  \
0          F6655E39465C2FF5B016980D918EA028   
1  FGJKJJ1_2BA0D0083976A5C1E3315413CDCFFCD2   
2          65018CD542145A3792BA09985734C12A   
3          6FAA4740F99408D4D2DDDD0B09BBDEFD   
4          F8437E44748D2C3FCF84019766F4E6DC   

                                             Imports     Timestamp  
0  [0.0, 0.0, 0.0, 0.018760568151320798, 0.0, 0.0...  1.263576e+09  
1  [0.0, 0.0, 0.0, 0.015410508958257556, 0.0, 0.0...  1.242321e+09  
2  [0.0, 0.0, 0.0, 0.015410508958257556, 0.0, 0.0...  1.195430e+09  
3  [0.0, 0.0, 0.0, 0.015410508958257556, 0.0, 0.0...  1.242321e+09  
4  [0.0, 0.0, 0.0, 0.015410508958257556, 0.0, 0.0...  1.242321e+09  


In [43]:
df_new.to_csv('transformed.csv')